In [4]:
import import_ipynb
from BollingerBandsIndicator import BollingerBandsIndicator
from EMAIndicator import EMAIndicator
from RSIIndicator import RSIIndicator
import pandas as pd
import datetime
from ResearchClass import StrategyTemplate
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from itertools import combinations
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
from ResearchClass import PlotEvaluations, EvaluationMetrics, TradesBook


In [ ]:
class MLModel(StrategyTemplate):
    def __init__(self, ticker_list, start_date, end_date, interval, indicator_parameters=None):
        if indicator_parameters is None:
            indicator_parameters = {
                'RSI': [14, 0.02],  # Lookback period, stop-loss percentage
                'EMA': [20, 0.02],  # Lookback period, stop-loss percentage
                'Bollinger Bands': [20, 2, 0.015]  # Lookback period, Std Dev multiplier, stop-loss
            }
        # Pass indicator_parameters to the parent class

        super().__init__(ticker_list, start_date, end_date, interval, indicator_parameters)
        self.rf_regressor = RandomForestRegressor(random_state=16)
        self.indicator_combinations = []
        self.best_combination = None
        self.data = pd.read_parquet(ticker_list[0])  # Load the first file

        self.start_date = start_date
        self.end_date = end_date
        self.interval = interval

        self.data['Close'] = pd.to_numeric(self.data['Close'], errors='coerce')  # Convert to numeric, replace invalid values with NaN
        self.data = self.data.dropna(subset=['Close'])  # Drop rows where 'Close' is NaN
        self.data['Return'] = self.data['Close'].pct_change()


    def create_combination(self):
        # List of indicators to test
        indicators = ['RSI', 'EMA', 'Bollinger Bands']
        # Generate all combinations of indicators
        indicator_combinations = []
        for r in range(1, len(indicators) + 1):
            indicator_combinations.extend(combinations(indicators, r))
        self.indicator_combinations = indicator_combinations

    def evaluate_combination(self, data, combination, parameters):
        """
        Evaluates a specific combination of indicators on the dataset.

        Parameters:
        - data: The stock data.
        - combination: A tuple of selected indicators.
        - parameters: A dictionary of parameters for each indicator.

        Returns:
        - Performance metric (e.g., total return).
        """
        # Reset data for fresh calculation
        data = data.copy()

        # Apply selected indicators
        if 'RSI' in combination:
            rsi = RSIIndicator(data, self.start_date, self.end_date, self.interval, parameters['RSI'])
            rsi.AddIndicators()
            data['RSI'] = rsi.data['RSI']

        if 'EMA' in combination:
            ema = EMAIndicator(data, self.start_date, self.end_date, self.interval, parameters['EMA'])
            ema.AddIndicators()
            data['EMA'] = ema.data['EMA']

        if 'Bollinger Bands' in combination:
            bollinger = BollingerBandsIndicator(data, self.start_date, self.end_date, self.interval, parameters['Bollinger Bands'])
            bollinger.AddIndicators()
            data['BB_UPPER'] = bollinger.data['BB_UPPER']
            data['BB_LOWER'] = bollinger.data['BB_LOWER']
            data['BB_MIDDLE'] = bollinger.data['BB_MIDDLE']

        # Implement a simple strategy for testing (e.g., buy/sell logic)
        data['Signal'] = 0
        if 'RSI' in combination:
            data.loc[data['RSI'] < 30, 'Signal'] = 1  # Buy
            data.loc[data['RSI'] > 60, 'Signal'] = -1  # Sell

        if 'EMA' in combination:
            data.loc[data['Close'] > data['EMA'], 'Signal'] = 1  # Buy
            data.loc[data['Close'] < data['EMA'], 'Signal'] = -1  # Sell

        if 'Bollinger Bands' in combination:
            data.loc[data['Close'] < data['BB_LOWER'], 'Signal'] = 1  # Buy
            data.loc[data['Close'] > data['BB_UPPER'], 'Signal'] = -1  # Sell

        # Calculate returns
        data['Daily_Return'] = data['Signal'].shift(1) * data['Close'].pct_change()
        total_return = data['Daily_Return'].sum()

        return total_return

    def BestCombo(self, data):
        # Example parameters for indicators
        parameters = {
            'RSI': [14, 0.02],  # Lookback period, stop-loss percentage
            'EMA': [20, 0.02],  # Lookback period, stop-loss percentage
            'Bollinger Bands': [20, 2, 0.015]  # Lookback period, Std Dev multiplier, stop-loss
        }

        best_performance = float('-inf')

        for combination in self.indicator_combinations:
            performance = self.evaluate_combination(data, combination, parameters)
            print(f"Combination: {combination}, Performance: {performance:.2f}")
            if performance > best_performance:
                best_performance = performance
                self.best_combination = combination

        print(f"Best Combination: {self.best_combination}, Best Performance: {best_performance:.2f}")

    def AddIndicators(self):
        # Debug: Check if 'Returns' column exists
        
       
        # Add indicators to the data
        if self.best_combination is None:
            raise ValueError("best_combination is not set. Call BestCombo() first.")

        if 'RSI' in self.best_combination:
            rsi = RSIIndicator(self.data, self.start_date, self.end_date, self.interval, self.indicator_parameters['RSI'])
            self.data['RSI'] = rsi.AddIndicators()
        
        
        if 'EMA' in self.best_combination:
            ema = EMAIndicator(self.data, self.start_date, self.end_date, self.interval, self.indicator_parameters['EMA'])
            self.data['EMA'] =  ema.AddIndicators()
        
        if 'Bollinger Bands' in self.best_combination:
            bollinger = BollingerBandsIndicator(self.data, self.start_date, self.end_date, self.interval, self.indicator_parameters['Bollinger Bands'])
            RETURN= bollinger.AddIndicators()
            self.data['BB_UPPER'] = RETURN[0]
            self.data['BB_LOWER'] = RETURN[1]
            self.data['BB_MIDDLE'] = RETURN[2]
        
        print(self.data.columns)
        # Prepare dataset for the ML model
        self.data_cleaned = self.data.fillna(0)
        X = self.data_cleaned[[col for col in self.data.columns if col in self.best_combination]]
        y = self.data_cleaned['Return']  # Target variable
        

        # Train the model
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=16)
        self.rf_regressor.fit(X_train, y_train)

    def strategyLogic(self, TradeBook, row, idx):
        # Prepare the input for the ML model
        input_data = row[[col for col in self.data.columns if col in self.best_combination]].values.reshape(1, -1)

        # Predict the return for the current time step
        predicted_return = self.rf_regressor.predict(input_data)[0]

        # Trading logic based on the predicted return
        if not TradeBook.PositionOpen():
            if predicted_return > 0:  # Predicts a positive return, open a long position
                TradeBook.OpenTrade(row['Close'], 1, row['Close'] * 0.98, idx, row['Close'] * 1.02)
        else:
            # Close the position if the predicted return is negative or after a certain condition
            if predicted_return < 0 or (idx - TradeBook.GetOpenOrderData()[3]) > 10:  # Example condition
                TradeBook.CloseTrade(row['Close'], idx)

# Example usage
ticker_list = ['data/MNQc1.parquet']
today = datetime.datetime.now()
start_date = str((today - datetime.timedelta(days=59)).strftime("%Y-%m-%d"))
end_date = str(today.strftime("%Y-%m-%d"))
interval = "1m"
data = pd.read_parquet(ticker_list[0])  # Load the first file

data['Close'] = pd.to_numeric(data['Close'], errors='coerce')  # Convert to numeric, replace invalid values with NaN
data = data.dropna(subset=['Close'])  # Drop rows where 'Close' is NaN
data['Return'] = data['Close'].pct_change()
if 'Return' not in data.columns or data['Return'].isna().all():
    data['Return'] = data['Close'].pct_change().fillna(0)



# Initialize and run the strategy
ml_model = MLModel(ticker_list, start_date, end_date, interval, None)
ml_model.create_combination()
print(data.columns)
ml_model.BestCombo(data)
print(data.columns)
TRADE_BOOK = ml_model.ApplyStrategyThroughTickers()

# Evaluate the strategy performance
bb_metrics = EvaluationMetrics(TRADE_BOOK, ticker_list, start_date, end_date)
bb_plots = PlotEvaluations(TRADE_BOOK, (10, 7))

# Print the evaluation metrics and plot the summary
bb_metrics.print_results()
bb_plots.PlotSummary()

Index(['#RIC', 'Date-Time', 'Open', 'High', 'Low', 'Close', 'Last', 'Volume',
       'No. Trades', 'Open Bid', 'High Bid', 'Low Bid', 'Close Bid',
       'No. Bids', 'Open Ask', 'High Ask', 'Low Ask', 'Close Ask', 'No. Asks',
       'Open Yld', 'High Yld', 'Low Yld', 'Close Yld', 'No. Ylds',
       'Open Bid Yld', 'High Bid Yld', 'Low Bid Yld', 'Close Bid Yld',
       'No. Bid Ylds', 'Open Ask Yld', 'High Ask Yld', 'Low Ask Yld',
       'Close Ask Yld', 'No. Ask Ylds', 'Open Zero Yld', 'High Zero Yld',
       'Low Zero Yld', 'Close Zero Yld', 'No. Zero Ylds', 'Open Bid Size',
       'High Bid Size', 'Low Bid Size', 'Close Bid Size', 'Open Ask Size',
       'High Ask Size', 'Low Ask Size', 'Close Ask Size', 'Return'],
      dtype='object')
Combination: ('RSI',), Performance: 0.37
Combination: ('EMA',), Performance: -1.34
Combination: ('Bollinger Bands',), Performance: 0.29
Combination: ('RSI', 'EMA'), Performance: -1.34
Combination: ('RSI', 'Bollinger Bands'), Performance: 0.49
Combinat

AssertionError: RSI: No Return Column Read from self.data!